# 06 — CG Index Calculation

Cleans raw category scores from `data/raw/numerical_indices.xlsx`, applies weighting and
imputation rules, merges in CINDEX from `data/raw/cindex_scores.xlsx`, and writes the
composite `data/processed/cg_scores.csv` consumed by `07_index_validation`,
`08_ff5_regression`, and `09_regression`.

## Sub-indices

| Category | What it measures | Scoring |
|---|---|---|
| **AINDEX** | Audit Committee — size, independence, meeting cadence, overlap with board meetings | Rule-based / numerical, extracted from XBRL filings |
| **BINDEX** | Board composition & conduct — size, independence, CEO duality, overboarding, attendance, ID tenure | Rule-based / numerical, extracted from XBRL filings |
| **DINDEX** | Disclosure — compensation disclosure (rule-based, D1) plus the *quality* of four board/RPT/whistleblower/insider-trading policies (D2–D5), which are **NLP-scored** by reading the policy text itself | Mixed: 1 rule-based item + 4 NLP-scored items |
| **OINDEX** | Ownership structure — promoter/DII/FII shareholding, pledge %, ownership concentration | Rule-based / numerical, extracted from XBRL filings |
| **TRINDEX** | Transparency & remuneration — director/CEO pay disclosure, RPT transactions, NRC governance | Rule-based / numerical, extracted from XBRL filings |
| **CINDEX** | Overall quality of the CG report's narrative content | **Fully NLP-scored** on report text (`data/raw/cindex_scores.xlsx`) — every item is derived from language-model scoring of free text, not from structured/XBRL fields matched against fixed rules. It is not a vendor score, and its provenance is not directly comparable to the other five sub-indices, which are rule-based except for DINDEX's four quality items noted above. |

Every sub-index is combined into per-firm-quarter category scores below; a mean composite,
weighted composite, and PCA-1 composite are all reported downstream (see `Avg_Score*`
columns and the PCA-1 section).

In [1]:
import pandas as pd
from pathlib import Path

BASE_DIR = Path.cwd()

In [2]:
df = pd.read_excel(BASE_DIR / "data" / "raw" / "numerical_indices.xlsx")
df["period_end"] = pd.to_datetime(df["period_end"])
print(df.shape)
df.head()

(197800, 11)


,file,source_type,company_name,symbol,reporting_quarter,period_end,ID,Metric,Category,Score,Weights
0,543257_543257_1872023182652_CG.xml,XML,Indian Railway Finance Corporation Limited,IRFC,Quarterly,2023-06-30,B1,Optimum Board size,BINDEX,0,3
1,543257_543257_1872023182652_CG.xml,XML,Indian Railway Finance Corporation Limited,IRFC,Quarterly,2023-06-30,B2,CEO duality,BINDEX,0,1
2,543257_543257_1872023182652_CG.xml,XML,Indian Railway Finance Corporation Limited,IRFC,Quarterly,2023-06-30,B3,Tenure of CEO,BINDEX,1,1
3,543257_543257_1872023182652_CG.xml,XML,Indian Railway Finance Corporation Limited,IRFC,Quarterly,2023-06-30,B4,Chairman Independence,BINDEX,0,1
4,543257_543257_1872023182652_CG.xml,XML,Indian Railway Finance Corporation Limited,IRFC,Quarterly,2023-06-30,B5,At least 50% NED in Board,BINDEX,1,1


In [3]:
def to_indian_fiscal_quarter(date):
    m, y = date.month, date.year
    if m <= 3:       # Jan–Mar → Q4 of same FY (e.g. Mar 2024 → Q4FY24)
        q, fy = 4, y
    elif m <= 6:     # Apr–Jun → Q1 (e.g. Jun 2023 → Q1FY24)
        q, fy = 1, y + 1
    elif m <= 9:     # Jul–Sep → Q2
        q, fy = 2, y + 1
    else:            # Oct–Dec → Q3
        q, fy = 3, y + 1
    return f"Q{q}FY{str(fy)[2:]}"

df["Q_FY"] = df["period_end"].apply(to_indian_fiscal_quarter)

In [4]:
matched = pd.read_excel(BASE_DIR / "data" / "processed" / "matched_companies.xlsx",
                        usecols=["BSE Code", "NSE Symbol"])

# Pass 1: normalise both sides (strip whitespace + uppercase) before matching
df["_sym_key"] = df["symbol"].str.strip().str.upper()
matched["_nse_key"] = matched["NSE Symbol"].str.strip().str.upper()

# Pass 2: manual overrides for renamed / inconsistent symbols
# Format: scores symbol (upper, stripped) → NSE Symbol key in matched_companies
SYMBOL_OVERRIDES = {
    "AMARAJABAT":           "ARE&M",        # renamed to Amara Raja Energy & Mobility
    "ANGELBRKG":            "ANGELONE",     # Angel Broking → Angel One
    "DAAWAT":               "LTFOODS",      # Daawat → LT Foods
    "GMRINFRA":             "GMRAIRPORT",   # GMR Infra → GMR Airports
    "GVT&D":                "GET&D",        # GE T&D India
    "IIFLWAM":              "360ONE",       # IIFL Wealth → 360 One WAM
    "KALPATPOWR":           "KPIL",         # Kalpataru Power → Kalpataru Projects
    "L&TFH":                "LTF",          # L&T Finance Holdings → L&T Finance
    "LAXMIMACH":            "LMW",          # Lakshmi Machine Works
    "MCDOWELL-N":           "UNITDSPR",     # McDowell's → United Spirits
    "MOTHERSUMI":           "MOTHERSON",    # Motherson Sumi → Samvardhana Motherson
    "RADICO KHAITAN LIMITED": "RADICO",     # full name used instead of symbol
    "RPOWER":               "RPOWER",       # case fix (Rpower → RPOWER)
    "SUVENPHAR":            "SUVENPHARMA",  # Suven Pharma
    "TATAMOTORS":           "TMCV",         # Tata Motors (CV segment listing)
    "VINTIORGA":            "VINATIORGA",   # Vinati Organics
}

df["_sym_key"] = df["_sym_key"].replace(SYMBOL_OVERRIDES)

# Merge on normalised keys
bse_map = matched.set_index("_nse_key")["BSE Code"].to_dict()
df["BSE Code"] = df["_sym_key"].map(bse_map)
df = df.drop(columns="_sym_key")

# Drop rows that could not be matched to a BSE Code
dropped_syms = df.loc[df["BSE Code"].isna(), "symbol"].dropna().unique()
print(f"Dropping {df['BSE Code'].isna().sum():,} unmatched rows for: {sorted(dropped_syms)}")
df = df[df["BSE Code"].notna()].reset_index(drop=True)

# Report
print(f"Matched:   {len(df):,} rows")


Dropping 4,171 unmatched rows for: ['CENTURYTEX', 'COHANCE', 'ETERNAL', 'JUBILANT', 'NOTLISTED', 'RUCHI', 'SUNCLAYLTD', 'TMPV']
Matched:   193,629 rows


In [5]:
unmatched_mask = df["BSE Code"].isna() & df["symbol"].notna()

df[unmatched_mask].groupby("symbol")["period_end"].agg(
    periods="nunique",
    first="min",
    last="max",
).sort_values("periods", ascending=False)

,periods,first,last
symbol,,,


In [6]:
# Metrics with specific data-gap placeholders (not NaN, not Prowess/NLP errors)
gap_scores = ["Needs Annual Report", "Needs revenue data", "Needs median employee pay", "Auditor opinion undetermined"]

gap_mask = df["Score"].isin(gap_scores)

df[gap_mask].groupby(["ID", "Metric", "Score"])["period_end"].agg(
    count="count",
    first="min",
    last="max",
).reset_index()

,ID,Metric,Score,count,first,last
0,A8,Auditor's Opinion,Auditor opinion undetermined,98,2019-06-30,2025-12-31
1,B7,Former CEOs should not be board members,Needs Annual Report,4503,2019-06-30,2025-12-31
2,O4,ID Shareholding,Needs Annual Report,4503,2019-06-30,2025-12-31
3,TR1,Director remuneration as a % of revenue,Needs revenue data,4503,2019-06-30,2025-12-31
4,TR2,CEO remuneration as a % of revenue,Needs revenue data,4503,2019-06-30,2025-12-31
5,TR4,CEO pay to median employee's pay ratio,Needs median employee pay,4503,2019-06-30,2025-12-31


In [7]:
# Drop metrics with data-gap placeholders or scoring failures
# B14: Board Meetings Attendance — all zeros from Q3FY25 onward (data collection failure)
# O4: ID Shareholding — 100% placeholder ("Needs Annual Report"), never scored (see 09_index_validation.ipynb §1.3)
# Also drop metrics identified as having negative within-firm return association
# (Entity FE t < -1.0 from per-metric panel regression in analysis.ipynb)
drop_ids = ["B7", "TR1", "TR2", "TR4", "A8", "B14", "O4"] # Data quality drops
before = len(df)
df = df[~df["ID"].isin(drop_ids)].reset_index(drop=True)
print(f"Dropped {before - len(df):,} rows for IDs: {drop_ids}")
print(f"Remaining: {len(df):,} rows")


Dropped 31,521 rows for IDs: ['B7', 'TR1', 'TR2', 'TR4', 'A8', 'B14', 'O4']
Remaining: 162,108 rows


In [8]:
# Companies with 'No Prowess match' in Score
no_prowess_mask = df["Score"].astype(str).str.strip() == "No Prowess match"

df[no_prowess_mask].groupby("symbol")["company_name"].first().reset_index().sort_values("symbol")

,symbol,company_name
0,AFFLE,AFFLE (INDIA) LIMITED
1,GET&D,GE T&D India Limited
2,GVT&D,GE T&D India Limited
3,HATSUN,HATSUN AGRO PRODUCT LIMITED
4,ISEC,ICICI Securities Limited
5,JINDALSTEL,JINDAL STEEL & POWER LIMITED
6,KANSAINER,KANSAI NEROLAC PAINTS LIMITED
7,LAXMIMACH,LAKSHMI MACHINE WORKS LIMITED
8,LMW,LMW LIMITED
9,PEL,Piramal Enterprises Limited


In [9]:
# Drop all companies with any 'No Prowess match' scores
no_prowess_symbols = df.loc[df["Score"].astype(str).str.strip() == "No Prowess match", "symbol"].unique()
before = len(df)
df = df[~df["symbol"].isin(no_prowess_symbols)].reset_index(drop=True)
print(f"Dropped symbols: {sorted(no_prowess_symbols)}")
print(f"Dropped {before - len(df):,} rows")
print(f"Remaining: {len(df):,} rows")

Dropped symbols: ['AFFLE', 'GET&D', 'GVT&D', 'HATSUN', 'ISEC', 'JINDALSTEL', 'KANSAINER', 'LAXMIMACH', 'LMW', 'PEL', 'SUNDRMFAST', 'SUVENPHAR', 'TATAMOTORS', 'UNIONBANK', 'VINATIORGA', 'VINTIORGA', 'ZOMATO']
Dropped 8,820 rows
Remaining: 153,288 rows


In [10]:
# Add Industry column from industry_map.xlsx
industry_map = pd.read_excel(BASE_DIR / "data" / "processed" / "industry_map.xlsx",
                             usecols=["BSE Code", "Industry"])

industry_map = (industry_map.dropna(subset=["BSE Code"])
                            .drop_duplicates(subset="BSE Code")
                            .set_index("BSE Code")["Industry"]
                            .to_dict())

df["Industry"] = df["BSE Code"].map(industry_map).fillna("Unknown Industry")

print(df["Industry"].value_counts().to_string())

Industry
Drug Manufacturers - Specialty & Generic    10692
Credit Services                              9108
Auto Parts                                   8244
Information Technology Services              6624
Banks - Regional                             5940
Electrical Equipment & Parts                 5688
Asset Management                             5580
Engineering & Construction                   4464
Specialty Chemicals                          4392
Medical Care Facilities                      4356
Aerospace & Defense                          3960
Specialty Industrial Machinery               3888
Building Materials                           3708
Steel                                        3672
Agricultural Inputs                          3636
Real Estate - Development                    3456
Auto Manufacturers                           3384
Household & Personal Products                3024
Building Products & Equipment                2844
Packaged Foods                           

## Score provenance: real vs. NLP-imputed vs. missing-source

Before any placeholder is overwritten, tag each row's provenance. This lets the composite
be rebuilt two ways further down: treating a scraper failure ("No policy URL found") as a
substantive 0, or as a missing observation excluded from that firm-quarter-metric average.
Conflating the two overstates governance failure for firms whose policy simply wasn't
found by the scraper.

In [ ]:
df["_source_flag"] = "real"
df.loc[df["Score"].astype(str).str.strip() == "NLP scoring failed", "_source_flag"] = "nlp_imputed"
df.loc[df["Score"].astype(str).str.strip() == "No policy URL found", "_source_flag"] = "missing_source"

print(df["_source_flag"].value_counts().to_string())

In [11]:
# Replace 'No policy URL found' scores with 0
mask = df["Score"].astype(str).str.strip() == "No policy URL found"
df.loc[mask, "Score"] = 0

print(f"Replaced {mask.sum():,} 'No policy URL found' scores with 0")

Replaced 2,473 'No policy URL found' scores with 0


In [12]:
# Impute 'NLP scoring failed' with median Score grouped by (ID, Industry)
nlp_mask = df["Score"].astype(str).str.strip() == "NLP scoring failed"
print(f"NLP scoring failed rows: {nlp_mask.sum():,}")

# Convert Score to numeric — NLP failed rows become NaN
df["Score"] = pd.to_numeric(df["Score"], errors="coerce")

# Compute median per (ID, Industry) from valid scores
industry_medians = df.groupby(["ID", "Industry"])["Score"].median()

# Fallback: overall median per ID (in case no industry match)
id_medians = df.groupby("ID")["Score"].median()

def impute_nlp(row):
    key = (row["ID"], row["Industry"])
    if key in industry_medians.index:
        return industry_medians[key]
    return id_medians.get(row["ID"], 0)

df.loc[nlp_mask, "Score"] = df[nlp_mask].apply(impute_nlp, axis=1)

print(f"Still NaN after imputation: {df['Score'].isna().sum()}")
print(f"\nImputed score distribution for NLP rows:")
print(df.loc[nlp_mask, ["ID", "Metric", "Industry", "Score"]].groupby(["ID", "Metric"])["Score"].describe().round(3))

NLP scoring failed rows: 8,429


Still NaN after imputation: 5835

Imputed score distribution for NLP rows:
                                               count   mean    std  min  25%  \
ID Metric                                                                      
D2 Quality of whistleblower policy            1670.0  0.383  0.244  0.0  0.0   
D3 Quality of policy against insider trading  1741.0  0.380  0.256  0.0  0.0   
D4 Quality of RPT policy                      1775.0  0.312  0.203  0.0  0.0   
D5 Quality of Board Evaluation Policy         1543.0  0.319  0.189  0.0  0.4   

                                              50%  75%  max  
ID Metric                                                    
D2 Quality of whistleblower policy            0.4  0.6  0.8  
D3 Quality of policy against insider trading  0.4  0.6  0.8  
D4 Quality of RPT policy                      0.4  0.4  0.6  
D5 Quality of Board Evaluation Policy         0.4  0.4  0.6  


In [ ]:
# ── SEBI weight scheme ───────────────────────────────────────────────────────
# `cg_scores_formulas.xlsx` classifies every metric as Mandates / Recommends /
# Over-the-Mandate, but its Comments column documents pass/fail rationale per
# metric, not a relative-importance multiplier — there is no citable source
# for a specific weight ratio between the three tiers. The {1, 2, 4} weights
# previously used here were unsourced. Absent a citation, the primary
# composite uses a linear weight vector that reflects only the ordinal
# severity SEBI's own categories imply: Mandates=1, Recommends=2,
# Over-the-Mandate=3. The old {1, 2, 4} scheme is kept as a robustness weight
# (`Weight_robust`) and produces a parallel composite (`Avg_Score_altweight`).
formulas = pd.read_excel(BASE_DIR / "data" / "raw" / "cg_scores_formulas.xlsx", usecols=["ID", "SEBI"])
sebi_map = formulas.dropna(subset=["ID"]).set_index("ID")["SEBI"].to_dict()
df["SEBI"] = df["ID"].map(sebi_map)

weight_map        = {"Mandates": 1, "Recommends": 2, "Over the Mandate": 3}  # primary — linear
weight_map_robust = {"Mandates": 1, "Recommends": 2, "Over the Mandate": 4}  # robustness — prior scheme

df["Weight"]        = df["SEBI"].map(weight_map).fillna(3).astype(int)
df["Weight_robust"] = df["SEBI"].map(weight_map_robust).fillna(4).astype(int)

print(df.groupby(["SEBI", "Weight", "Weight_robust"]).size().reset_index(name="count").to_string(index=False))

In [ ]:
# Score_na / Weight_na: alternative treatment of scraper failures. `Score`/`Weight`
# above treat "No policy URL found" (`missing_source`) as 0 — current production
# behaviour, preserved for `Avg_Score`. `Score_na`/`Weight_na` instead drop those
# observations from the weighted average entirely, so a firm's composite reflects
# only what was actually observed, not an assumed governance failure.
df["Score_na"]  = df["Score"].where(df["_source_flag"] != "missing_source")
df["Weight_na"] = df["Weight"].where(df["Score_na"].notna())

print(df.groupby("_source_flag").size().rename("rows").to_string())

In [ ]:
# ── Resolve duplicate (BSE Code, period_end, ID) rows ────────────────────────
# 09_index_validation.ipynb found 2,833 duplicate (symbol, period_end, ID) groups:
# 2,736 are exact repeats (same score — harmless) and 97 genuinely conflict
# (different scores from different source files for the same filing period,
# e.g. 3M India's B6 score for Q1FY23 is 0.7454 in one file vs 0.9129 in
# another). Both cases currently enter groupby(...).sum() as if they were two
# independent metrics, silently double-weighting that ID's contribution to
# the category average. Collapse to one row per (BSE Code, period_end, ID) by
# averaging Score — a no-op for exact repeats, a neutral resolution for
# genuine conflicts. Score_na/Weight_na are averaged the same way (mean skips
# NaN, so a row that's missing_source in one file but real in another keeps
# the real value).
dupe_key = ["BSE Code", "period_end", "ID"]
n_groups = df.groupby(dupe_key).ngroups
before = len(df)

agg = {col: "first" for col in df.columns if col not in dupe_key + ["Score", "Score_na", "Weight_na"]}
agg["Score"]     = "mean"
agg["Score_na"]  = "mean"
agg["Weight_na"] = "mean"
df = df.groupby(dupe_key, as_index=False).agg(agg)

print(f"Collapsed {before - len(df):,} duplicate rows down to {n_groups:,} unique (BSE Code, period_end, ID) groups")
print(f"Remaining: {len(df):,} rows")

In [ ]:
df['weighted_score']     = df['Score']    * df['Weight']
df['weighted_score_na']  = df['Score_na'] * df['Weight_na']
df['weighted_score_alt'] = df['Score']    * df['Weight_robust']

# Proper weighted average: sum(Score × Weight) / sum(Weight) → range [0, 1]
# min_count=1 makes sum() return NaN (not 0) when every row in a group is NaN,
# so a firm-quarter-metric where every observation was missing_source yields
# a NaN Avg_Score_missing_as_na rather than a spurious 0/0 → NaN via divide.
grp = df.groupby(["BSE Code", "Q_FY", "Category"])
category_scores = pd.concat([
    grp["weighted_score"].sum(min_count=1)     / grp["Weight"].sum(min_count=1),
    grp["weighted_score_na"].sum(min_count=1)  / grp["Weight_na"].sum(min_count=1),
    grp["weighted_score_alt"].sum(min_count=1) / grp["Weight_robust"].sum(min_count=1),
], axis=1)
category_scores.columns = ["Avg_Score", "Avg_Score_missing_as_na", "Avg_Score_altweight"]
category_scores["Avg_Score_missing_as_zero"] = category_scores["Avg_Score"]
category_scores = category_scores[["Avg_Score", "Avg_Score_missing_as_zero",
                                    "Avg_Score_missing_as_na", "Avg_Score_altweight"]].reset_index()

n_diff = (category_scores["Avg_Score"].round(6) != category_scores["Avg_Score_missing_as_na"].round(6)).sum()
print(f"Score range: {category_scores['Avg_Score'].min():.4f} – {category_scores['Avg_Score'].max():.4f}")
print(f"Firm-quarter-categories where missing-source treatment changes the average: {n_diff:,} / {len(category_scores):,}")
print(category_scores.shape)
category_scores.head(10)

In [ ]:
# ── Load and process CINDEX, then append to category_scores ─────────────────
ci = pd.read_excel(BASE_DIR / 'data' / 'raw' / 'cindex_scores.xlsx')
ci = ci.rename(columns={'BSE_code': 'BSE Code'})
ci['BSE Code']   = pd.to_numeric(ci['BSE Code'], errors='coerce')
ci['period_end'] = pd.to_datetime(ci['period_end'], errors='coerce')
ci['Score']      = pd.to_numeric(ci['Score'], errors='coerce')

# Map period_end → Q_FY (Indian FY: Apr-Mar)
def period_to_qfy(dt):
    if pd.isna(dt): return None
    fy   = dt.year if dt.month <= 3 else dt.year + 1
    q    = {1:'Q4',2:'Q4',3:'Q4',4:'Q1',5:'Q1',6:'Q1',
            7:'Q2',8:'Q2',9:'Q2',10:'Q3',11:'Q3',12:'Q3'}[dt.month]
    return f"{q}FY{str(fy)[2:]}"

ci['Q_FY'] = ci['period_end'].apply(period_to_qfy)
ci = ci.dropna(subset=['Q_FY','Score','BSE Code'])

# Filter to matched_companies universe only
matched_codes = set(matched['BSE Code'].dropna())
before = ci['BSE Code'].nunique()
ci = ci[ci['BSE Code'].isin(matched_codes)]
print(f"CINDEX: filtered from {before} → {ci['BSE Code'].nunique()} companies (matched universe only)")

# Simple (unweighted) average of all CINDEX sub-metrics. CINDEX has no SEBI
# weight scheme and no "No policy URL found" placeholder, so the
# missing-source / alt-weight variants collapse to the same value here.
ci_scores = (
    ci.groupby(['BSE Code','Q_FY','Category'])['Score'].mean()
).reset_index().rename(columns={'Score': 'Avg_Score'})
ci_scores['Avg_Score_missing_as_zero'] = ci_scores['Avg_Score']
ci_scores['Avg_Score_missing_as_na']   = ci_scores['Avg_Score']
ci_scores['Avg_Score_altweight']       = ci_scores['Avg_Score']

print(f"CINDEX rows  : {len(ci_scores)}")
print(f"CINDEX Q_FYs : {sorted(ci_scores['Q_FY'].unique())}")
print(f"Score range  : {ci_scores['Avg_Score'].min():.4f} – {ci_scores['Avg_Score'].max():.4f}")

# Append to main category_scores
category_scores = pd.concat([category_scores, ci_scores], ignore_index=True)
print(f"\nTotal rows after merge: {len(category_scores)}")
print(f"Categories: {sorted(category_scores['Category'].unique())}")

## PCA-1 composite

Data-driven alternative to the mean/weighted composite. Each of the six sub-indices is
standardised *cross-sectionally within its own Q_FY* (so a quarter with a different firm
mix or scoring vintage doesn't distort the loadings), then the first principal component
is extracted across the six standardised sub-indices. Reported as a robustness alongside
the mean and SEBI-weighted composites, not a replacement — saved as its own row with
`Category = "PCA1"`, using `Avg_Score` for the PC1 value (the missing-source / alt-weight
variants don't apply to a PCA composite and are left blank for these rows).

In [17]:
# Save to CSV
out_path = BASE_DIR / "data" / "processed" / "cg_scores.csv"
category_scores.to_csv(out_path, index=False)
print(f"Saved → {out_path}")

Saved → /Users/jaganathapandiyan/Desktop/Python/governance-and-stock-performance/data/processed/cg_scores.csv
